In [ ]:
!pip install -U langgraph langchain-google-genai langchain-core pydantic

^C


   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/750.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/750.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/750.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/750.9 kB ? eta -:--:--
   -------------------------- ----------- 524.3/750.9 kB 349.7 kB/s eta 0:00:01
   -------------------------- ----------- 524.3/750.9 kB 349.7 kB/s eta 0:00:01
   -------------------------- ----------- 524.3/750.9 kB 349.7 kB/s eta 0:00:01
   ---------------------------------------- 

In [ ]:
import os
import json
import getpass
from typing import TypedDict, List, Dict, Any
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# GLOBAL SETUP (Done exactly once)
# ==========================================
print("--- INITIALIZING GLOBAL SETUP ---")
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Google AI Studio Key: ")

# Initialize the Model globally
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)


# ==========================================
# AGENT 1: SMART SCHEDULER (With Human-In-The-Loop)
# ==========================================
print("\n=== SETTING UP SCHEDULER AGENT ===")

# --- Long Term Memory Setup ---
MEMORY_FILE = "scheduler_memory.json"

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return []

def save_memory(new_feedback):
    history = load_memory()
    history.append(new_feedback)
    with open(MEMORY_FILE, "w") as f:
        json.dump(history, f)
    print("💾 Agent has learned this preference for next time.")

# --- State & Pydantic Models ---
class SchedulerState(TypedDict):
    tasks: List[Dict]
    calendar_slots: List[str]
    energy_map: Dict[str, str]
    schedule: Dict[str, Any]
    user_feedback: str

class EnergyMapOutput(BaseModel):
    mapping: Dict[str, str] = Field(
        description="A dictionary mapping exact time slots to 'High Energy', 'Medium Energy', or 'Low Energy'."
    )

# --- Nodes ---
def infer_energy_node(state: SchedulerState):
    print("⚡ Agent is analyzing energy patterns & past memory...")
    slots = state["calendar_slots"]
    past_feedback = load_memory()

    prompt = f"""
    You are an Energy Analyzer. Map these time slots to 'High Energy', 'Medium Energy', or 'Low Energy'.
    DEFAULT LOGIC: Morning is High, Afternoon is Low.
    CRITICAL - USER PAST FEEDBACK (Override default if needed): {past_feedback}
    TIME SLOTS TO ANALYZE: {slots}
    """

    structured_llm = llm.with_structured_output(EnergyMapOutput)
    try:
        response = structured_llm.invoke(prompt)
        energy_map = response.mapping
    except Exception as e:
        print(f"Fallback triggered due to mapping error: {e}")
        energy_map = {s: "Medium Energy" for s in slots}

    return {"energy_map": energy_map}

def schedule_node(state: SchedulerState):
    print("📅 Agent is matching tasks to slots...")
    tasks = sorted(state["tasks"], key=lambda x: x['priority'])
    energy_map = state["energy_map"]
    remaining_slots = state["calendar_slots"].copy()
    final_schedule = {}

    for task in tasks:
        dur = task.get('duration', 1)
        if len(remaining_slots) < dur:
            print(f"⚠️ Not enough slots left for task: {task['name']}")
            continue

        assigned_slots = []
        if task['priority'] == 1:
            high_slots = [s for s in remaining_slots if "High" in energy_map.get(s, "")]
            if len(high_slots) >= dur:
                assigned_slots = high_slots[:dur]
            else:
                assigned_slots = remaining_slots[:dur]
        else:
            assigned_slots = remaining_slots[:dur]

        for slot in assigned_slots:
            final_schedule[slot] = task
            remaining_slots.remove(slot)

    return {"schedule": final_schedule}

def review_node(state: SchedulerState):
    print("\n--- 🤖 AGENT PROPOSED SCHEDULE ---")
    for time, task in state["schedule"].items():
        energy_context = state['energy_map'].get(time, "Unknown")
        print(f"[{time}] {task['name']} \t--> ({energy_context})")
    print("-----------------------------------\n⏸️ Graph is pausing for human feedback...")
    return {}

def learning_node(state: SchedulerState):
    feedback = state.get("user_feedback", "")
    if len(feedback) > 3 and "good" not in feedback.lower() and "ok" not in feedback.lower():
        save_memory(feedback)
    else:
        print("👍 Schedule accepted. No new constraints to learn.")
    return {}

# --- Graph Compilation ---
scheduler_workflow = StateGraph(SchedulerState)
scheduler_workflow.add_node("energy_analyzer", infer_energy_node)
scheduler_workflow.add_node("scheduler", schedule_node)
scheduler_workflow.add_node("human_review", review_node)
scheduler_workflow.add_node("learner", learning_node)

scheduler_workflow.add_edge(START, "energy_analyzer")
scheduler_workflow.add_edge("energy_analyzer", "scheduler")
scheduler_workflow.add_edge("scheduler", "human_review")
scheduler_workflow.add_edge("human_review", "learner")
scheduler_workflow.add_edge("learner", END)

memory_saver = MemorySaver()
scheduler_app = scheduler_workflow.compile(
    checkpointer=memory_saver,
    interrupt_before=["learner"]
)


# ==========================================
# AGENT 2: PROJECT TRACKER
# ==========================================
print("\n=== SETTING UP PROJECT TRACKER ===")

class ProjectState(TypedDict):
    transcript: str
    action_items: str
    blockers: str
    final_report: str

def extractor_node(state: ProjectState):
    prompt = f"Extract a bulleted list of Action Items (Who, What, Due Date) from: {state['transcript']}"
    return {"action_items": llm.invoke(prompt).content}

def risk_node(state: ProjectState):
    prompt = f"Identify blocks/risks in this text: {state['transcript']}. If none, say 'None'."
    return {"blockers": llm.invoke(prompt).content}

def report_node(state: ProjectState):
    report = f"🚀 REPORT:\n\nACTIONS:\n{state['action_items']}\n\nRISKS:\n{state['blockers']}"
    return {"final_report": report}

tracker_workflow = StateGraph(ProjectState)
tracker_workflow.add_node("extractor", extractor_node)
tracker_workflow.add_node("risk", risk_node)
tracker_workflow.add_node("report", report_node)

tracker_workflow.add_edge(START, "extractor")
tracker_workflow.add_edge("extractor", "risk")
tracker_workflow.add_edge("risk", "report")
tracker_workflow.add_edge("report", END)
tracker_app = tracker_workflow.compile()


# ==========================================
# AGENT 3: EMAIL TRIAGE
# ==========================================
print("\n=== SETTING UP EMAIL BUTLER ===")

class EmailButlerState(TypedDict):
    email_content: Dict[str, str]
    is_spam: bool
    detected_style: str
    draft_response: str
    execution_log: str

class SpamDetection(BaseModel):
    is_spam: bool = Field(description="Return True if the email is spam, marketing, or a scam. False if it is a legitimate message.")

def triage_officer_node(state: EmailButlerState):
    print("🔍 Triage Officer: Asking Gemini to analyze intent...")
    email = state["email_content"]
    prompt = f"Analyze this email. Subject: {email.get('subject')}\nBody: {email.get('body')}"

    structured_triage = llm.with_structured_output(SpamDetection)
    response = structured_triage.invoke(prompt)
    return {"is_spam": response.is_spam}

def style_mimic_node(state: EmailButlerState):
    print("✍️ Style Mimic: Analyzing communication style...")
    prompt = f"Describe the writer's tone in 5 words or less: {state['email_content'].get('body')}"
    return {"detected_style": llm.invoke(prompt).content.strip()}

def draft_architect_node(state: EmailButlerState):
    print("📝 Draft Architect: Generating response...")
    email, style = state["email_content"], state["detected_style"]
    prompt = f"Write a helpful reply. Sender: {email.get('sender')}. Topic: {email.get('subject')}. Tone to mimic: {style}."
    return {"draft_response": llm.invoke(prompt).content}

def sql_adaptation_node(state: EmailButlerState):
    print("💾 SQL Agent: Logging transaction...")
    status = "BLOCKED (Spam)" if state["is_spam"] else "PROCESSED"
    return {"execution_log": f"Status: {status}"}

def email_routing_logic(state: EmailButlerState):
    return "logger" if state["is_spam"] else "mimic"

email_workflow = StateGraph(EmailButlerState)
email_workflow.add_node("triage", triage_officer_node)
email_workflow.add_node("mimic", style_mimic_node)
email_workflow.add_node("architect", draft_architect_node)
email_workflow.add_node("logger", sql_adaptation_node)

email_workflow.add_edge(START, "triage")
email_workflow.add_conditional_edges("triage", email_routing_logic)
email_workflow.add_edge("mimic", "architect")
email_workflow.add_edge("architect", "logger")
email_workflow.add_edge("logger", END)
email_app = email_workflow.compile()


# ==========================================
# EXECUTIONS & DEMOS
# ==========================================
if __name__ == "__main__":

    print("\n\n" + "="*50)
    print("▶️ DEMO 1: SCHEDULER (WITH HUMAN-IN-THE-LOOP)")
    print("="*50)

    scheduler_input = {
        "tasks": [
            {"name": "Deep Coding", "priority": 1, "duration": 2},
            {"name": "Email Catchup", "priority": 2, "duration": 1},
        ],
        "calendar_slots": ["9:00", "10:00", "13:00", "14:00"],
        "energy_map": {}, "schedule": {}, "user_feedback": ""
    }

    config = {"configurable": {"thread_id": "test_user_1"}}

    print("▶ Run Phase 1: Planning...")
    scheduler_app.invoke(scheduler_input, config)

    print("\n(Simulated Human Input): 'I hate 9am meetings.'")
    simulated_input = "I hate 9am meetings."

    scheduler_app.update_state(config, {"user_feedback": simulated_input})
    print("▶ Run Phase 2: Resuming and Learning...")
    scheduler_app.invoke(None, config)


    print("\n\n" + "="*50)
    print("▶️ DEMO 2: EMAIL TRIAGE (SPAM VS LEGIT)")
    print("="*50)

    spam_email = {
        "sender": "Prize Bot",
        "subject": "YOU WON $1,000,000",
        "body": "Click here immediately to claim your massive cash prize!!! This is TRUE."
    }
    print("\n--- Testing Obvious Spam ---")
    email_res = email_app.invoke({"email_content": spam_email, "is_spam": False, "detected_style": "", "draft_response": "", "execution_log": ""})
    print(f"Result: {email_res['execution_log']}")

    legit_email = {
        "sender": "John",
        "subject": "Matrix Pointers",
        "body": "Hey, I'm stuck on pointer arithmetic. Can you help?"
    }
    print("\n--- Testing Legit Email ---")
    email_res = email_app.invoke({"email_content": legit_email, "is_spam": False, "detected_style": "", "draft_response": "", "execution_log": ""})
    print(f"Result: {email_res['execution_log']}")
    print(f"Draft: {email_res['draft_response']}")

    print("\n\n" + "="*50)
    print("▶️ DEMO 3: PROJECT TRACKER")
    print("="*50)

    tracker_input = {
        "transcript": "Update: Alice is handling the DB migration (due Friday). Bob is locked out of the server.",
        "action_items": "",
        "blockers": "",
        "final_report": ""
    }

    print("▶ Running Project Tracker...")
    tracker_res = tracker_app.invoke(tracker_input)
    print(f"\n{tracker_res['final_report']}")

--- INITIALIZING GLOBAL SETUP ---

=== SETTING UP SCHEDULER AGENT ===

=== SETTING UP PROJECT TRACKER ===

=== SETTING UP EMAIL BUTLER ===


▶️ DEMO 1: SCHEDULER (WITH HUMAN-IN-THE-LOOP)
▶ Run Phase 1: Planning...
⚡ Agent is analyzing energy patterns & past memory...
📅 Agent is matching tasks to slots...

--- 🤖 AGENT PROPOSED SCHEDULE ---
[9:00] Deep Coding 	--> (Low Energy)
[10:00] Deep Coding 	--> (High Energy)
[13:00] Email Catchup 	--> (Low Energy)
-----------------------------------
⏸️ Graph is pausing for human feedback...

(Simulated Human Input): 'I hate 9am meetings.'
▶ Run Phase 2: Resuming and Learning...
💾 Agent has learned this preference for next time.


▶️ DEMO 2: EMAIL TRIAGE (SPAM VS LEGIT)

--- Testing Obvious Spam ---
🔍 Triage Officer: Asking Gemini to analyze intent...
💾 SQL Agent: Logging transaction...
Result: Status: BLOCKED (Spam)

--- Testing Legit Email ---
🔍 Triage Officer: Asking Gemini to analyze intent...
✍️ Style Mimic: Analyzing communication style...
📝 